# KION: как учитывать долю просмотра фильма

**Один автономный запуск:** 5 способов обработки взаимодействий × 3 seed × 2 временных среза = **30 моделей**. Компактная BERT4Rec, S=20, фиксированное маскирование 35%, по 20 000 шагов. Все модели обучаются с нуля.

В Kaggle включите **Internet** и **GPU** (T4 подходит). Затем **Run All**. При двух T4 используется одна — cuda:0. Установка иной версии PyTorch и подключение Google Drive не требуются.

| Метод | История и обучение |
| --- | --- |
| ALL | Все допустимые просмотры, равные веса |
| DELETE10 | Просмотры ≤10% удалены из истории до выбора последних объектов |
| TARGET10 | Слабые просмотры остаются в контексте, но их цели имеют нулевой вес |
| STEP10 | Вес 1 для ≤10%, иначе 3 |
| SOFT | Вес 0,2 + 0,8 × доля просмотра от 0 до 1 |

Основная оценка — NDCG@10 по будущим фильмам с долей просмотра ≥50%. Также сохраняются Recall@10 и метрики любого просмотра. Доля просмотра — поведенческий признак, не оценка удовлетворённости. Данные агрегированы по последней дате.

Ноутбук сохраняет прогресс, проверяет совместимость продолжения и выдаёт один ZIP. **Сохраняйте также исходные выходы Kaggle с весами** — они нужны при переносе в новую сессию.


## 1. Загрузка встроенного кода

В этой ячейке находятся все функции исследования. Она самодостаточна: файлы проекта и прежних ноутбуков не нужны. Код проверяет контрольную сумму исходного KION, разбиение, параметры и сохранённые модели. Ячейку можно оставить свёрнутой; изменять её для запуска не требуется.


In [ ]:
import hashlib
PROGRAM_SOURCE = '"""Standalone KION study 2: film watch-percentage handling, five policies, two cuts.\n\nThe notebook embeds this file verbatim. No imports from article-1 implementations.\n"""\n# %% runtime\nimport os\nos.environ.setdefault(\'CUBLAS_WORKSPACE_CONFIG\', \':4096:8\')\nos.environ.setdefault(\'MPLBACKEND\', \'Agg\')\nimport hashlib\nimport io\nimport json\nimport math\nimport platform\nimport random\nimport shutil\nimport time\nimport urllib.request\nimport zipfile\nfrom dataclasses import asdict, dataclass, field\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\n\nVERSION = \'kion-watch-feedback-1.0\'\nARCHIVE_SHA256 = \'fddc5b2215bfff49bcfa21e464441f1fc6dc30f8f4afe41d3f23bf3fc45f3542\'\nSOURCE_URL = \'https://raw.githubusercontent.com/irsafilo/KION_DATASET/f69775be31fa5779907cf0a92ddedb70037fb5ae/data_en.zip\'\nMETHODS = [\'ALL\', \'DELETE10\', \'TARGET10\', \'STEP10\', \'SOFT\']\nSEEDS = [17, 29, 43]\nCUTS = [(\'main\', \'2021-08-01\', \'2021-08-08\', \'2021-08-15\'),\n        (\'replication\', \'2021-07-18\', \'2021-07-25\', \'2021-08-01\')]\nEXPECTED = {\n    \'main\': {\'eligible_users\': 81545, \'train_pairs\': 1085046, \'catalog_items\': 10131,\n             \'input_positions\': 783385, \'validation_users_any\': 15523, \'validation_users_high\': 8471},\n    \'replication\': {\'eligible_users\': 66620, \'train_pairs\': 1069327, \'catalog_items\': 10298,\n                    \'input_positions\': 780135, \'validation_users_any\': 16058, \'validation_users_high\': 9650},\n}\n\n@dataclass(frozen=True)\nclass WatchConfig:\n    n_users: int = 50000\n    min_history: int = 10\n    sample_seed: int = 20261031\n    sequence_length: int = 20\n    hidden_size: int = 64\n    layers: int = 2\n    heads: int = 2\n    dropout: float = .1\n    mask_probability: float = .35\n    weak_threshold: float = 10.0\n    high_threshold: float = 50.0\n    soft_floor: float = .2\n    batch_size: int = 128\n    eval_batch_size: int = 128\n    learning_rate: float = .001\n    weight_decay: float = .01\n    gradient_clip: float = 5.0\n    max_steps: int = 20000\n    eval_every: int = 1000\n    checkpoint_every: int = 1000\n    bootstrap_reps: int = 5000\n    bootstrap_seed: int = 20261031\n    technical_mode: bool = False\n\n    def validate(self):\n        if not self.technical_mode and self != WatchConfig():\n            raise ValueError(\'Научные настройки отличаются от согласованного протокола. Не меняйте их для продолжения серии.\')\n        if self.hidden_size % self.heads or not 0 < self.mask_probability < 1:\n            raise ValueError(\'Неверные размерности или вероятность маскирования.\')\n        if min(self.n_users, self.batch_size, self.eval_batch_size, self.max_steps,\n               self.eval_every, self.checkpoint_every, self.bootstrap_reps) < 1 or self.sequence_length < 2:\n            raise ValueError(\'Неверный бюджет или длина входа.\')\n        return self\n\ndef canonical(value):\n    return hashlib.sha256(json.dumps(value, sort_keys=True, ensure_ascii=False, allow_nan=False).encode()).hexdigest()\n\ndef file_hash(path):\n    digest = hashlib.sha256()\n    with Path(path).open(\'rb\') as f:\n        for chunk in iter(lambda: f.read(1024 * 1024), b\'\'):\n            digest.update(chunk)\n    return digest.hexdigest()\n\ndef array_hash(array):\n    a = np.ascontiguousarray(array)\n    return hashlib.sha256(str((a.dtype.str, a.shape)).encode() + a.tobytes()).hexdigest()\n\ndef save_json(path, value):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temporary = path.with_name(path.name + \'.tmp\')\n    temporary.write_text(json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False) + \'\\n\', encoding=\'utf-8\')\n    temporary.replace(path)\n\ndef save_csv(path, rows):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temporary = path.with_name(path.name + \'.tmp\')\n    (rows if isinstance(rows, pd.DataFrame) else pd.DataFrame(rows)).to_csv(temporary, index=False)\n    temporary.replace(path)\n\ndef save_torch(path, state):\n    path = Path(path)\n    temporary = path.with_name(path.name + \'.tmp\')\n    torch.save(state, temporary)\n    temporary.replace(path)\n\ndef load_torch(path):\n    # All stored states contain only tensors and primitive containers. Never unpickle arbitrary objects.\n    return torch.load(path, map_location=\'cpu\', weights_only=True)\n\ndef seed_everything(seed):\n    random.seed(seed)\n    np.random.seed(seed)\n    torch.manual_seed(seed)\n    if torch.cuda.is_available():\n        torch.cuda.manual_seed_all(seed)\n    torch.use_deterministic_algorithms(True)\n    torch.backends.cudnn.benchmark = False\n    torch.backends.cudnn.allow_tf32 = False\n    torch.backends.cuda.matmul.allow_tf32 = False\n    torch.set_float32_matmul_precision(\'highest\')\n\ndef select_device(cfg):\n    cfg.validate()\n    if not torch.cuda.is_available() and not cfg.technical_mode:\n        raise RuntimeError(\'GPU не обнаружена. В Kaggle включите Accelerator → GPU и перезапустите сессию.\')\n    return torch.device(\'cuda:0\' if torch.cuda.is_available() and not cfg.technical_mode else \'cpu\')\n\ndef environment(device):\n    return {\'python\': platform.python_version(), \'torch\': torch.__version__, \'numpy\': np.__version__,\n            \'pandas\': pd.__version__, \'cuda\': torch.version.cuda, \'device\': device.type,\n            \'gpu\': torch.cuda.get_device_name(device) if device.type == \'cuda\' else None,\n            \'precision\': \'float32\', \'torch_cpu_threads\': torch.get_num_threads(), \'implementation\': VERSION}\n\ndef sync(device):\n    if device.type == \'cuda\':\n        torch.cuda.synchronize(device)\n\ndef acquire_archive(cache, explicit=\'\'):\n    cache = Path(cache)\n    cache.mkdir(parents=True, exist_ok=True)\n    path = Path(explicit) if explicit else cache / \'data_en.zip\'\n    if not path.exists():\n        if explicit:\n            raise FileNotFoundError(\'Указанный архив KION не найден: \' + str(path))\n        temporary = path.with_suffix(\'.download\')\n        print(\'Скачивание закреплённого архива KION…\', flush=True)\n        urllib.request.urlretrieve(SOURCE_URL, temporary)\n        if file_hash(temporary) != ARCHIVE_SHA256:\n            raise ValueError(\'Контрольная сумма скачанного KION не совпала.\')\n        temporary.replace(path)\n    if file_hash(path) != ARCHIVE_SHA256:\n        raise ValueError(\'Нужен официальный архив data_en.zip закреплённой ревизии; этот файл отличается.\')\n    return path\n\n# %% data\n@dataclass\nclass EvalExample:\n    user_id: int\n    history: list\n    delete_history: list\n    seen: list\n    targets_any: list\n    targets_high: list\n    history_length: int\n    target_day: str\n\n@dataclass\nclass SplitData:\n    name: str\n    train_end: str\n    validation_end: str\n    test_end: str\n    users: np.ndarray\n    item_ids: np.ndarray\n    tokens: dict\n    percentages: dict\n    training_indices: dict\n    validation: list\n    manifest: dict\n    rows: pd.DataFrame = field(repr=False)\n\ndef read_films(archive):\n    with zipfile.ZipFile(archive) as z:\n        with z.open(\'data_en/items_en.csv\') as f:\n            items = pd.read_csv(f, usecols=[\'item_id\', \'content_type\'])\n        with z.open(\'data_en/interactions.csv\') as f:\n            rows = pd.read_csv(f, dtype={\'user_id\': \'int32\', \'item_id\': \'int32\', \'watched_pct\': \'float32\'})\n    if not items.item_id.is_unique or rows.duplicated([\'user_id\', \'item_id\']).any():\n        raise ValueError(\'Нарушена уникальность объектов или агрегированных пар пользователь–объект.\')\n    film_ids = items.loc[items.content_type.eq(\'film\'), \'item_id\']\n    films = rows.loc[rows.item_id.isin(film_ids)].copy()\n    if (films.watched_pct.dropna().lt(0) | films.watched_pct.dropna().gt(100)).any():\n        raise ValueError(\'Доля просмотра вне 0–100: требуется проверка исходных данных.\')\n    return films.sort_values([\'user_id\', \'last_watch_dt\', \'item_id\']).reset_index(drop=True)\n\ndef pack_histories(frame, users, item_index, sequence_length):\n    tokens = np.zeros((len(users), sequence_length), dtype=np.int64)\n    percentages = np.zeros((len(users), sequence_length), dtype=np.float32)\n    user_index = {int(u): i for i, u in enumerate(users)}\n    latest = frame.groupby(\'user_id\', sort=False).tail(sequence_length)\n    for user_id, group in latest.groupby(\'user_id\', sort=False):\n        i, n = user_index[int(user_id)], len(group)\n        tokens[i, :n] = [item_index[int(item)] for item in group.item_id]\n        percentages[i, :n] = group.watched_pct.to_numpy(dtype=np.float32)\n    return tokens, percentages\n\ndef make_examples(rows, users, item_ids, start, end, cfg):\n    item_index = {int(item): i + 1 for i, item in enumerate(item_ids)}\n    past_raw = rows.loc[rows.last_watch_dt.le(start)]\n    past = past_raw.loc[past_raw.watched_pct.notna() & past_raw.item_id.isin(item_index)]\n    histories, deleted, seen = {}, {}, {}\n    for user, group in past.groupby(\'user_id\', sort=False):\n        mapped = [item_index[int(i)] for i in group.item_id]\n        histories[int(user)] = mapped\n        deleted[int(user)] = [item_index[int(i)] for i in group.loc[group.watched_pct.gt(cfg.weak_threshold), \'item_id\']]\n    for user, group in past_raw.groupby(\'user_id\', sort=False):\n        seen[int(user)] = sorted({item_index[int(i)] for i in group.item_id if int(i) in item_index})\n    future = rows.loc[rows.last_watch_dt.gt(start) & rows.last_watch_dt.le(end)]\n    first_day = future.groupby(\'user_id\').last_watch_dt.transform(\'min\')\n    first = future.loc[future.last_watch_dt.eq(first_day)]\n    result, lost = [], 0\n    for user, group in first.groupby(\'user_id\', sort=True):\n        user = int(user)\n        known = group.loc[group.item_id.isin(item_index) & group.watched_pct.notna()]\n        already = set(seen.get(user, []))\n        any_targets = sorted({item_index[int(i)] for i in known.item_id} - already)\n        high_targets = sorted({item_index[int(i)] for i in known.loc[known.watched_pct.ge(cfg.high_threshold), \'item_id\']} - already)\n        if not any_targets:\n            lost += 1\n            continue\n        hist, dhist = histories.get(user, []), deleted.get(user, [])\n        result.append(EvalExample(user, hist[-(cfg.sequence_length - 1):], dhist[-(cfg.sequence_length - 1):],\n                                  seen.get(user, []), any_targets, high_targets, len(hist), str(group.last_watch_dt.iloc[0])))\n    if not result or not any(x.targets_high for x in result):\n        raise ValueError(\'Нет общей оценочной когорты или целей высокой доли просмотра.\')\n    manifest = {\'start_exclusive\': start, \'end_inclusive\': end,\n                \'users_with_raw_future_films\': int(future.user_id.nunique()), \'users_without_known_targets\': lost,\n                \'users_any\': len(result), \'users_high\': sum(bool(x.targets_high) for x in result),\n                \'targets_any\': sum(len(x.targets_any) for x in result),\n                \'targets_high\': sum(len(x.targets_high) for x in result),\n                \'fingerprint\': canonical([asdict(x) for x in result])}\n    return result, manifest\n\ndef prepare_split(films, cut, cfg):\n    cfg.validate()\n    name, train_end, validation_end, test_end = cut\n    valid_train = films.loc[films.last_watch_dt.le(train_end) & films.watched_pct.notna()]\n    counts = valid_train.groupby(\'user_id\').size()\n    eligible = np.sort(counts.loc[counts.ge(cfg.min_history)].index.to_numpy())\n    if len(eligible) < cfg.n_users:\n        raise ValueError(\'Недостаточно пользователей с принятой историей; минимум автоматически не понижается.\')\n    users = np.sort(np.random.default_rng(cfg.sample_seed).choice(eligible, cfg.n_users, replace=False))\n    rows = films.loc[films.user_id.isin(users)].copy()\n    training = valid_train.loc[valid_train.user_id.isin(users)]\n    item_ids = np.sort(training.item_id.unique())\n    item_index = {int(item): i + 1 for i, item in enumerate(item_ids)}\n    tokens, percentages, indices = {}, {}, {}\n    for key, subset in [(\'all\', training), (\'delete\', training.loc[training.watched_pct.gt(cfg.weak_threshold)])]:\n        tokens[key], percentages[key] = pack_histories(subset, users, item_index, cfg.sequence_length)\n        lengths = np.count_nonzero(tokens[key], axis=1)\n        indices[key] = np.flatnonzero(lengths >= 2)\n        if not len(indices[key]):\n            raise ValueError(\'После фильтрации нет обучающих историй с целью и видимым объектом.\')\n    validation, val_manifest = make_examples(rows, users, item_ids, train_end, validation_end, cfg)\n    stats = {\'eligible_users\': len(eligible), \'selected_users\': len(users), \'train_pairs\': len(training),\n             \'catalog_items\': len(item_ids), \'input_positions\': int(np.count_nonzero(tokens[\'all\'])),\n             \'validation_users_any\': len(validation), \'validation_users_high\': sum(bool(x.targets_high) for x in validation),\n             \'delete_training_users\': len(indices[\'delete\']), \'delete_removed_training_users\': len(users) - len(indices[\'delete\']),\n             \'weak_input_fraction\': float((percentages[\'all\'][tokens[\'all\'] > 0] <= cfg.weak_threshold).mean())}\n    if not cfg.technical_mode:\n        for key, expected in EXPECTED[name].items():\n            if stats[key] != expected:\n                raise ValueError(f\'Срез {name}, {key}: {stats[key]} вместо {expected}. Обучение не начато.\')\n    manifest = {\'split\': name, \'train_end\': train_end, \'validation_end\': validation_end, \'test_end\': test_end,\n                \'stats\': stats, \'validation\': val_manifest,\n                \'array_hashes\': {\'users\': array_hash(users), \'items\': array_hash(item_ids),\n                     **{k + \'_tokens\': array_hash(a) for k, a in tokens.items()},\n                     **{k + \'_pct\': array_hash(a) for k, a in percentages.items()}},\n                \'source_archive_sha256\': ARCHIVE_SHA256}\n    manifest[\'fingerprint\'] = canonical(manifest)\n    return SplitData(name, train_end, validation_end, test_end, users, item_ids, tokens, percentages,\n                     indices, validation, manifest, rows)\n\ndef make_plan(data, cfg, device, code_hash):\n    plan = {\'version\': VERSION, \'mode\': \'technical_smoke_not_scientific\' if cfg.technical_mode else \'scientific\',\n            \'config\': asdict(cfg), \'methods\': METHODS, \'seeds\': SEEDS, \'cuts\': [list(c) for c in CUTS],\n            \'code_sha256\': code_hash, \'source_archive_sha256\': ARCHIVE_SHA256,\n            \'environment\': environment(device), \'data\': {k: v.manifest for k, v in data.items()},\n            \'models\': len(data) * len(METHODS) * len(SEEDS),\n            \'checkpoint_selection\': \'validation_ndcg10_high; first maximum; no test selection\',\n            \'primary_contrasts\': [[s, \'SOFT\', right, \'ndcg10_high\'] for s in data for right in [\'ALL\', \'STEP10\']],\n            \'primary_interval_level\': .9875, \'other_intervals\': \'descriptive_95_percent\',\n            \'test_use\': \'fixed_protocol; KION previously used for another study; no claim of independent dataset\',\n            \'loss\': \'sum(w*CE)/sum(w); mask p=.35; ZERO and missing pct are different\',\n            \'limits\': \'aggregated last-date film interactions; conditional user bootstrap; three training seeds\'}\n    return {\'plan_hash\': canonical(plan), \'plan\': plan}\n\ndef queue(data):\n    return [(split, method, seed) for seed in SEEDS for method in METHODS for split in data]\n\ndef run_name(split, method, seed):\n    return f\'{split}__{method}__seed{seed}\'\n\ndef freeze_plan(root, frozen, data):\n    root = Path(root)\n    root.mkdir(parents=True, exist_ok=True)\n    target = root / \'watch_study_plan.json\'\n    if target.exists():\n        old = json.loads(target.read_text(encoding=\'utf-8\'))\n        if old != frozen:\n            differences = [k for k in frozen[\'plan\'] if old.get(\'plan\', {}).get(k) != frozen[\'plan\'][k]]\n            raise ValueError(\'Сохранённая серия отличается: \' + \', \'.join(differences) + \'. Проверку не отключайте.\')\n    else:\n        if any((root / folder).exists() and any((root / folder).iterdir()) for folder in [\'runs\', \'test_runs\']):\n            raise ValueError(\'Найдены результаты без корневого плана; автоматическое смешивание запрещено.\')\n        save_json(target, frozen)\n    for name, split in data.items():\n        folder = root / \'data\' / name\n        folder.mkdir(parents=True, exist_ok=True)\n        save_json(folder / \'manifest.json\', split.manifest)\n        np.savez_compressed(folder / \'selection_catalog.npz\', users=split.users, item_ids=split.item_ids)\n    return root\n\ndef restore_folder(destination, frozen, source=\'\', search_root=\'/kaggle/input\'):\n    destination = Path(destination)\n    if (destination / \'watch_study_plan.json\').exists():\n        return None\n    if source:\n        candidates = [Path(source) / \'watch_study_plan.json\']\n        if not candidates[0].is_file():\n            raise FileNotFoundError(\'RESUME_FROM: укажите папку серии, содержащую watch_study_plan.json и runs.\')\n    else:\n        candidates = list(Path(search_root).rglob(\'watch_study_plan.json\')) if Path(search_root).exists() else []\n    matching = []\n    for path in candidates:\n        try:\n            saved = json.loads(path.read_text(encoding=\'utf-8\'))\n        except (OSError, ValueError):\n            continue\n        if saved == frozen:\n            matching.append(path.parent)\n        elif source:\n            raise ValueError(\'Указанная папка восстановления имеет другую среду, данные, код или параметры.\')\n    if not matching:\n        return None\n    # Prefer the snapshot with the most complete models; never merge incompatible snapshots.\n    chosen = max(matching, key=lambda p: (len(list((p / \'runs\').glob(\'*/completed.json\'))),\n                                         len(list((p / \'test_runs\').glob(\'*/completed.json\'))), str(p)))\n    if destination.exists() and any(destination.iterdir()):\n        raise ValueError(\'Папка назначения не пуста и не имеет плана: восстановление не перезаписывает файлы.\')\n    shutil.copytree(chosen, destination, dirs_exist_ok=True)\n    print(\'Восстановлены сохранённые выходы:\', chosen, flush=True)\n    return chosen\n\n# %% model_training\ndef check_execution(cfg, frozen, device):\n    if asdict(cfg) != frozen[\'plan\'][\'config\'] or environment(device) != frozen[\'plan\'][\'environment\']:\n        raise ValueError(\'Во время серии изменены настройки или программная среда.\')\n    if METHODS != frozen[\'plan\'][\'methods\'] or SEEDS != frozen[\'plan\'][\'seeds\']:\n        raise ValueError(\'Изменены согласованные методы или seed.\')\n\nclass CompactBERT4Rec(nn.Module):\n    """Same compact ID-only architecture as article 1; no pretrained language model."""\n    def __init__(self, n_items, cfg):\n        super().__init__()\n        self.n_items, self.mask_id = n_items, n_items + 1\n        self.item_embedding = nn.Embedding(n_items + 2, cfg.hidden_size, padding_idx=0)\n        self.position_embedding = nn.Embedding(cfg.sequence_length, cfg.hidden_size)\n        self.input_norm = nn.LayerNorm(cfg.hidden_size)\n        self.dropout = nn.Dropout(cfg.dropout)\n        block = nn.TransformerEncoderLayer(cfg.hidden_size, cfg.heads, 4 * cfg.hidden_size, cfg.dropout,\n                    activation=\'gelu\', batch_first=True, norm_first=False)\n        self.encoder = nn.TransformerEncoder(block, cfg.layers, enable_nested_tensor=False)\n        self.output_dense = nn.Linear(cfg.hidden_size, cfg.hidden_size)\n        self.output_norm = nn.LayerNorm(cfg.hidden_size)\n        self.item_bias = nn.Parameter(torch.zeros(n_items))\n        for name, parameter in self.named_parameters():\n            if parameter.ndim >= 2:\n                nn.init.trunc_normal_(parameter, std=.02, a=-.04, b=.04)\n            elif name.endswith(\'weight\'):\n                nn.init.ones_(parameter)\n            else:\n                nn.init.zeros_(parameter)\n        with torch.no_grad():\n            self.item_embedding.weight[0].zero_()\n\n    def encode(self, tokens):\n        positions = torch.arange(tokens.shape[1], device=tokens.device)\n        h = self.dropout(self.input_norm(self.item_embedding(tokens) + self.position_embedding(positions)[None]))\n        return self.encoder(h, src_key_padding_mask=tokens.eq(0), is_causal=False)\n\n    def scores(self, h):\n        h = self.output_norm(F.gelu(self.output_dense(h)))\n        return F.linear(h, self.item_embedding.weight[1:self.n_items + 1], self.item_bias)\n\n    def masked_logits(self, tokens, mask):\n        return self.scores(self.encode(tokens)[mask])\n\n    def next_scores(self, tokens, positions):\n        h = self.encode(tokens)\n        return self.scores(h[torch.arange(len(tokens), device=tokens.device), positions])\n\ndef target_weights(pct, method, cfg):\n    if method in [\'ALL\', \'DELETE10\']:\n        return torch.ones_like(pct)\n    if method == \'TARGET10\':\n        return pct.gt(cfg.weak_threshold).to(pct.dtype)\n    if method == \'STEP10\':\n        return torch.where(pct.gt(cfg.weak_threshold), 3.0, 1.0)\n    if method == \'SOFT\':\n        return cfg.soft_floor + (1 - cfg.soft_floor) * pct / 100\n    raise ValueError(\'Неизвестный метод: \' + method)\n\ndef masked_batch(clean, pct, mask_id, cfg, seed, draw):\n    generator = torch.Generator(device=\'cpu\').manual_seed(seed * 1000003 + draw)\n    real = clean.ne(0)\n    lengths = real.sum(dim=1)\n    if lengths.min() < 2 or not torch.isfinite(pct[real]).all() or ((pct[real] < 0) | (pct[real] > 100)).any():\n        raise ValueError(\'Неверная история или доля просмотра в обучающем пакете.\')\n    selected = torch.rand(clean.shape, generator=generator).lt(cfg.mask_probability) & real\n    for row, count in enumerate(selected.sum(dim=1).tolist()):\n        n = int(lengths[row])\n        if count == 0:\n            selected[row, int(torch.randint(n, (1,), generator=generator))] = True\n        elif count == n:\n            selected[row, int(torch.randint(n, (1,), generator=generator))] = False\n    corrupted = clean.clone()\n    corrupted[selected] = mask_id\n    return corrupted, selected, clean[selected] - 1, pct[selected]\n\ndef weighted_loss(logits, labels, weights):\n    if weights.sum().item() <= 0:\n        return None\n    return (F.cross_entropy(logits, labels, reduction=\'none\') * weights).sum() / weights.sum()\n\ndef metrics(recommended, relevant):\n    if not relevant:\n        return None, None\n    rel = set(relevant)\n    hits = [int(i in rel) for i in recommended[:10]]\n    dcg = sum(x / math.log2(i + 2) for i, x in enumerate(hits))\n    ideal = sum(1 / math.log2(i + 2) for i in range(min(10, len(rel))))\n    return dcg / ideal, sum(hits) / len(rel)\n\ndef aggregate_users(rows):\n    frame = rows if isinstance(rows, pd.DataFrame) else pd.DataFrame(rows)\n    return {\'users_any\': len(frame), \'users_high\': int(frame.has_high.sum()),\n            **{m: float(frame[m].mean()) for m in [\'ndcg10_any\', \'recall10_any\', \'ndcg10_high\', \'recall10_high\']}}\n\n@torch.inference_mode()\ndef evaluate(model, examples, method, cfg, device):\n    model.eval()\n    rows = []\n    for offset in range(0, len(examples), cfg.eval_batch_size):\n        batch = examples[offset:offset + cfg.eval_batch_size]\n        tokens = torch.zeros((len(batch), cfg.sequence_length), dtype=torch.long)\n        positions = []\n        for row, ex in enumerate(batch):\n            h = ex.delete_history if method == \'DELETE10\' else ex.history\n            tokens[row, :len(h) + 1] = torch.tensor(h + [model.mask_id])\n            positions.append(len(h))\n        scores = model.next_scores(tokens.to(device), torch.tensor(positions, device=device))\n        for row, ex in enumerate(batch):\n            if ex.seen:\n                scores[row, torch.tensor(ex.seen, device=device) - 1] = -torch.inf\n        order = torch.argsort(scores, dim=1, descending=True, stable=True)[:, :min(10, model.n_items)]\n        finite = scores.gather(1, order).isfinite().cpu().tolist()\n        for ex, ids, allowed, context_length in zip(batch, order.cpu().tolist(), finite, positions):\n            rec = [i + 1 for i, ok in zip(ids, allowed) if ok]\n            na, ra = metrics(rec, ex.targets_any)\n            nh, rh = metrics(rec, ex.targets_high)\n            rows.append({\'user_id\': ex.user_id, \'history_length\': ex.history_length, \'context_length\': context_length,\n                         \'has_high\': bool(ex.targets_high), \'targets_any\': len(ex.targets_any), \'targets_high\': len(ex.targets_high),\n                         \'ndcg10_any\': na, \'recall10_any\': ra, \'ndcg10_high\': nh, \'recall10_high\': rh,\n                         \'recommendation_internal_ids\': \' \'.join(map(str, rec))})\n    return aggregate_users(rows), pd.DataFrame(rows)\n\ndef expected_identity(split, method, seed, frozen):\n    return {\'plan_hash\': frozen[\'plan_hash\'], \'split\': split.name, \'method\': method, \'seed\': seed,\n            \'data_fingerprint\': split.manifest[\'fingerprint\']}\n\ndef verify_completed(folder, identity):\n    folder = Path(folder)\n    marker = folder / \'completed.json\'\n    if not marker.exists():\n        return None\n    record = json.loads(marker.read_text(encoding=\'utf-8\'))\n    if record[\'identity\'] != identity:\n        raise ValueError(\'Несовместимый завершённый запуск: \' + str(folder))\n    for name, digest in record[\'checksums\'].items():\n        if not (folder / name).is_file() or file_hash(folder / name) != digest:\n            raise ValueError(\'Файл завершённого запуска отсутствует или изменён: \' + str(folder / name))\n    return record\n\ndef train_run(split, method, seed, cfg, root, frozen, device, stop_after_updates=0):\n    check_execution(cfg, frozen, device)\n    folder = Path(root) / \'runs\' / run_name(split.name, method, seed)\n    identity = expected_identity(split, method, seed, frozen)\n    if verify_completed(folder, identity):\n        return True\n    if (Path(root) / \'test_selection_lock.json\').exists():\n        raise ValueError(\'Тест уже зафиксирован: отсутствующую модель нельзя заменить новым обучением.\')\n    folder.mkdir(parents=True, exist_ok=True)\n    config_record = {\'identity\': identity, \'config\': asdict(cfg), \'environment\': environment(device)}\n    config_path = folder / \'config.json\'\n    if config_path.exists() and json.loads(config_path.read_text(encoding=\'utf-8\')) != config_record:\n        raise ValueError(\'Изменена конфигурация незавершённого запуска.\')\n    save_json(config_path, config_record)\n    seed_everything(seed)\n    model = CompactBERT4Rec(len(split.item_ids), cfg).to(device)\n    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.learning_rate, weight_decay=cfg.weight_decay)\n    step, draw, best_score, best_step = 0, 0, None, 0\n    training_seconds, evaluation_seconds, checkpoint_seconds = 0.0, 0.0, 0.0\n    history, best_metrics = [], None\n    totals = {\'real_positions\': 0, \'masked_positions\': 0, \'positive_weight_targets\': 0,\n              \'weight_sum\': 0.0, \'weight_square_sum\': 0.0, \'zero_weight_batches\': 0, \'weighted_loss_sum\': 0.0}\n    if (folder / \'last.pt\').exists():\n        state = load_torch(folder / \'last.pt\')\n        if state[\'identity\'] != identity:\n            raise ValueError(\'last.pt относится к другой серии.\')\n        model.load_state_dict(state[\'model\'])\n        optimizer.load_state_dict(state[\'optimizer\'])\n        step, draw, best_score, best_step = [state[k] for k in [\'step\', \'draw\', \'best_score\', \'best_step\']]\n        training_seconds, evaluation_seconds, checkpoint_seconds = [state[k] for k in [\'training_seconds\', \'evaluation_seconds\', \'checkpoint_seconds\']]\n        history, best_metrics, totals = state[\'history\'], state[\'best_metrics\'], state[\'totals\']\n        torch.set_rng_state(state[\'torch_rng\'])\n        if device.type == \'cuda\':\n            torch.cuda.set_rng_state_all(state[\'cuda_rng\'])\n        best_candidate = folder / f\'candidate_step_{best_step}.pt\'\n        if best_score is not None and (not best_candidate.exists() or file_hash(best_candidate) != state[\'best_sha256\']):\n            raise ValueError(\'Не совпадает лучший checkpoint незавершённого запуска.\')\n        print(f\'Продолжение {folder.name}: шаг {step}/{cfg.max_steps}\', flush=True)\n    else:\n        print(\'Обучение:\', folder.name, flush=True)\n\n    def checkpoint():\n        nonlocal checkpoint_seconds\n        t = time.perf_counter()\n        save_torch(folder / \'last.pt\', {\'identity\': identity, \'model\': model.state_dict(),\n            \'optimizer\': optimizer.state_dict(), \'step\': step, \'draw\': draw, \'best_score\': best_score,\n            \'best_step\': best_step, \'history\': history, \'best_metrics\': best_metrics, \'totals\': totals,\n            \'training_seconds\': training_seconds, \'evaluation_seconds\': evaluation_seconds,\n            \'checkpoint_seconds\': checkpoint_seconds, \'torch_rng\': torch.get_rng_state(),\n            \'cuda_rng\': torch.cuda.get_rng_state_all() if device.type == \'cuda\' else [],\n            \'best_sha256\': file_hash(folder / f\'candidate_step_{best_step}.pt\') if best_score is not None else None})\n        save_csv(folder / \'history.csv\', history)\n        # Old best snapshots are removed only AFTER last.pt atomically commits the new pointer.\n        for pattern, keep in [(\'candidate_step_*.pt\', f\'candidate_step_{best_step}.pt\'),\n                              (\'candidate_users_*.csv\', f\'candidate_users_{best_step}.csv\')]:\n            for candidate in folder.glob(pattern):\n                if candidate.name != keep:\n                    candidate.unlink()\n        checkpoint_seconds += time.perf_counter() - t\n\n    key = \'delete\' if method == \'DELETE10\' else \'all\'\n    valid_indices = split.training_indices[key]\n    tokens, pct = split.tokens[key], split.percentages[key]\n    start_step = step\n    if not (folder / \'last.pt\').exists():\n        checkpoint()\n    try:\n        while step < cfg.max_steps:\n            model.train()\n            sync(device)\n            t = time.perf_counter()\n            draw += 1\n            chosen = np.random.default_rng(seed * 1000003 + draw).choice(valid_indices, cfg.batch_size, replace=True)\n            clean, watch = torch.from_numpy(tokens[chosen]), torch.from_numpy(pct[chosen])\n            corrupted, selected, labels, target_pct = masked_batch(clean, watch, model.mask_id, cfg, seed, draw)\n            weights = target_weights(target_pct, method, cfg)\n            if weights.sum().item() == 0:\n                totals[\'zero_weight_batches\'] += 1\n                training_seconds += time.perf_counter() - t\n                if totals[\'zero_weight_batches\'] > 1000:\n                    raise ValueError(\'Слишком много пакетов без ненулевых целей; проверьте данные.\')\n                continue\n            optimizer.zero_grad(set_to_none=True)\n            loss = weighted_loss(model.masked_logits(corrupted.to(device), selected.to(device)), labels.to(device), weights.to(device))\n            if not torch.isfinite(loss):\n                raise FloatingPointError(\'Функция потерь не конечна.\')\n            loss.backward()\n            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.gradient_clip)\n            optimizer.step()\n            sync(device)\n            training_seconds += time.perf_counter() - t\n            step += 1\n            totals[\'real_positions\'] += int(clean.ne(0).sum())\n            totals[\'masked_positions\'] += int(selected.sum())\n            totals[\'positive_weight_targets\'] += int(weights.gt(0).sum())\n            totals[\'weight_sum\'] += float(weights.sum())\n            totals[\'weight_square_sum\'] += float(weights.square().sum())\n            totals[\'weighted_loss_sum\'] += float(loss.detach())\n            if step % cfg.eval_every == 0 or step == cfg.max_steps:\n                sync(device)\n                t = time.perf_counter()\n                val, users = evaluate(model, split.validation, method, cfg, device)\n                sync(device)\n                evaluation_seconds += time.perf_counter() - t\n                history.append({\'step\': step, \'training_seconds\': training_seconds, \'evaluation_seconds\': evaluation_seconds,\n                                \'mean_training_loss\': totals[\'weighted_loss_sum\'] / step, **val})\n                if best_score is None or val[\'ndcg10_high\'] > best_score:\n                    best_score, best_step, best_metrics = val[\'ndcg10_high\'], step, val\n                    t = time.perf_counter()\n                    save_torch(folder / f\'candidate_step_{step}.pt\', {\'identity\': identity, \'model\': model.state_dict(), \'step\': step,\n                                                  \'validation\': val})\n                    save_csv(folder / f\'candidate_users_{step}.csv\', users)\n                    checkpoint_seconds += time.perf_counter() - t\n                print(f\'{folder.name}: {step}/{cfg.max_steps}; val NDCG высокой доли={val["ndcg10_high"]:.5f}\', flush=True)\n            if step % cfg.checkpoint_every == 0 or step == cfg.max_steps:\n                checkpoint()\n            if stop_after_updates and step - start_step >= stop_after_updates and step < cfg.max_steps:\n                checkpoint()\n                return False\n    except KeyboardInterrupt:\n        # A signal can arrive between optimizer.step and counters; use the last completed atomic checkpoint.\n        print(\'Остановка. Сохранённый last.pt позволит повторить не более одного интервала шагов.\', flush=True)\n        return False\n    if best_metrics is None:\n        raise ValueError(\'Не выполнена валидация; завершение не подтверждено.\')\n    shutil.copyfile(folder / f\'candidate_step_{best_step}.pt\', folder / \'best.pt\')\n    shutil.copyfile(folder / f\'candidate_users_{best_step}.csv\', folder / \'best_validation_per_user.csv\')\n    result = {\'identity\': identity, \'completed_steps\': step, \'best_step\': best_step,\n              \'training_seconds\': training_seconds, \'evaluation_seconds\': evaluation_seconds,\n              \'checkpoint_seconds\': checkpoint_seconds, \'best_validation\': best_metrics,\n              \'actual_mask_fraction\': totals[\'masked_positions\'] / totals[\'real_positions\'],\n              \'effective_weighted_targets\': totals[\'weight_sum\'] ** 2 / totals[\'weight_square_sum\'],\n              \'training_users\': int(len(valid_indices)), \'totals\': totals, \'mode\': frozen[\'plan\'][\'mode\']}\n    save_json(folder / \'summary.json\', result)\n    required = [\'best.pt\', \'last.pt\', \'summary.json\', \'config.json\', \'history.csv\', \'best_validation_per_user.csv\']\n    save_json(folder / \'completed.json\', {\'identity\': identity, \'completed_steps\': step,\n                                        \'checksums\': {name: file_hash(folder / name) for name in required}})\n    del optimizer, model\n    if device.type == \'cuda\':\n        torch.cuda.empty_cache()\n    return True\n\ndef run_study(data, cfg, root, frozen, device, max_runs=0):\n    check_execution(cfg, frozen, device)\n    done, trained = 0, 0\n    jobs = queue(data)\n    for split_name, method, seed in jobs:\n        split = data[split_name]\n        folder = Path(root) / \'runs\' / run_name(split_name, method, seed)\n        identity = expected_identity(split, method, seed, frozen)\n        if verify_completed(folder, identity):\n            done += 1\n            continue\n        if (Path(root) / \'test_selection_lock.json\').exists():\n            raise ValueError(\'После фиксации теста отсутствует завершённая модель. Нужны исходные сохранённые выходы.\')\n        if max_runs and trained >= max_runs:\n            break\n        if not train_run(split, method, seed, cfg, root, frozen, device):\n            save_json(Path(root) / \'progress.json\', {\'plan_hash\': frozen[\'plan_hash\'], \'completed_models\': done,\n                       \'total_models\': len(jobs), \'next_run\': folder.name, \'status\': \'training_incomplete\'})\n            return False\n        trained += 1\n        done += 1\n        save_json(Path(root) / \'progress.json\', {\'plan_hash\': frozen[\'plan_hash\'], \'completed_models\': done,\n                   \'total_models\': len(jobs), \'last_completed\': folder.name, \'status\': \'training_complete\' if done == len(jobs) else \'training\'})\n    return done == len(jobs)\n\ndef benchmark(data, cfg, root, frozen, device, steps=200):\n    check_execution(cfg, frozen, device)\n    path = Path(root) / \'benchmark.json\'\n    if path.exists():\n        old = json.loads(path.read_text(encoding=\'utf-8\'))\n        if old[\'plan_hash\'] == frozen[\'plan_hash\']:\n            return old\n    split = next(iter(data.values()))\n    seed_everything(2718)\n    model = CompactBERT4Rec(len(split.item_ids), cfg).to(device)\n    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.learning_rate, weight_decay=cfg.weight_decay)\n    sync(device)\n    start = time.perf_counter()\n    for step in range(steps):\n        chosen = np.random.default_rng(2718 + step).choice(split.training_indices[\'all\'], cfg.batch_size)\n        clean, pct = torch.from_numpy(split.tokens[\'all\'][chosen]), torch.from_numpy(split.percentages[\'all\'][chosen])\n        x, mask, y, q = masked_batch(clean, pct, model.mask_id, cfg, 2718, step)\n        model.train()\n        optimizer.zero_grad(set_to_none=True)\n        loss = weighted_loss(model.masked_logits(x.to(device), mask.to(device)), y.to(device), torch.ones_like(q).to(device))\n        loss.backward()\n        torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.gradient_clip)\n        optimizer.step()\n    sync(device)\n    train_sec = time.perf_counter() - start\n    start = time.perf_counter()\n    evaluate(model, split.validation, \'ALL\', cfg, device)\n    sync(device)\n    val_sec = time.perf_counter() - start\n    estimated = len(queue(data)) * (cfg.max_steps * train_sec / steps + math.ceil(cfg.max_steps / cfg.eval_every) * val_sec)\n    result = {\'plan_hash\': frozen[\'plan_hash\'], \'scope\': \'timing_only_not_a_scientific_run\', \'steps\': steps,\n              \'training_seconds_per_step\': train_sec / steps, \'one_validation_seconds\': val_sec,\n              \'rough_hours_all_models\': estimated / 3600,\n              \'estimate_excludes_setup_checkpointing_test_and_bootstrap\': True}\n    save_json(path, result)\n    del optimizer, model\n    if device.type == \'cuda\':\n        torch.cuda.empty_cache()\n    return result\n\n# %% reports\ndef freeze_test(data, cfg, root, frozen):\n    root = Path(root)\n    hashes = {}\n    for split_name, method, seed in queue(data):\n        folder = root / \'runs\' / run_name(split_name, method, seed)\n        record = verify_completed(folder, expected_identity(data[split_name], method, seed, frozen))\n        if not record or record[\'completed_steps\'] != cfg.max_steps:\n            raise ValueError(\'Тест доступен только после завершения всех согласованных моделей.\')\n        hashes[folder.name] = record[\'checksums\'][\'best.pt\']\n    examples, manifests = {}, {}\n    for name, split in data.items():\n        examples[name], manifests[name] = make_examples(split.rows, split.users, split.item_ids,\n                                                       split.validation_end, split.test_end, cfg)\n    lock = {\'plan_hash\': frozen[\'plan_hash\'], \'best_checkpoint_sha256\': hashes, \'test_manifests\': manifests,\n            \'selection\': \'all_30_planned_models; validation_checkpoint_only\'}\n    path = root / \'test_selection_lock.json\'\n    if path.exists() and json.loads(path.read_text(encoding=\'utf-8\')) != lock:\n        raise ValueError(\'После фиксации теста изменены модели или оценочная когорта.\')\n    save_json(path, lock)\n    return examples, lock\n\ndef test_models(data, examples, cfg, root, frozen, device, lock):\n    root = Path(root)\n    runs = []\n    for split_name, method, seed in queue(data):\n        name = run_name(split_name, method, seed)\n        train_folder, folder = root / \'runs\' / name, root / \'test_runs\' / name\n        identity = expected_identity(data[split_name], method, seed, frozen)\n        test_identity = {**identity, \'weight_sha256\': lock[\'best_checkpoint_sha256\'][name],\n                         \'test_fingerprint\': lock[\'test_manifests\'][split_name][\'fingerprint\']}\n        marker = folder / \'completed.json\'\n        if marker.exists():\n            cached = json.loads(marker.read_text(encoding=\'utf-8\'))\n            if cached[\'identity\'] != test_identity:\n                raise ValueError(\'Изменён сохранённый тест: \' + name)\n            for file, digest in cached[\'checksums\'].items():\n                if not (folder / file).exists() or file_hash(folder / file) != digest:\n                    raise ValueError(\'Повреждён тестовый файл: \' + name + \'/\' + file)\n            test = json.loads((folder / \'summary.json\').read_text(encoding=\'utf-8\'))\n        else:\n            seed_everything(seed)\n            model = CompactBERT4Rec(len(data[split_name].item_ids), cfg).to(device)\n            state = load_torch(train_folder / \'best.pt\')\n            if state[\'identity\'] != identity:\n                raise ValueError(\'Несовместимый лучший checkpoint.\')\n            model.load_state_dict(state[\'model\'])\n            sync(device)\n            t = time.perf_counter()\n            result, users = evaluate(model, examples[split_name], method, cfg, device)\n            sync(device)\n            test = {\'identity\': test_identity, \'metrics\': result, \'test_seconds\': time.perf_counter() - t}\n            save_json(folder / \'summary.json\', test)\n            save_csv(folder / \'per_user.csv\', users)\n            save_json(marker, {\'identity\': test_identity, \'checksums\': {f: file_hash(folder / f) for f in [\'summary.json\', \'per_user.csv\']}})\n            del model\n        train = json.loads((train_folder / \'summary.json\').read_text(encoding=\'utf-8\'))\n        runs.append({\'split\': split_name, \'method\': method, \'seed\': seed, \'best_step\': train[\'best_step\'],\n                     **{\'validation_\' + k: v for k, v in train[\'best_validation\'].items()},\n                     **{\'test_\' + k: v for k, v in test[\'metrics\'].items()},\n                     **{k: train[k] for k in [\'training_seconds\', \'evaluation_seconds\', \'checkpoint_seconds\',\n                                              \'actual_mask_fraction\', \'effective_weighted_targets\', \'training_users\']},\n                     **{\'total_\' + k: v for k, v in train[\'totals\'].items()}})\n    result = pd.DataFrame(runs)\n    save_csv(root / \'results_per_run.csv\', result)\n    return result\n\ndef bootstrap_interval(values, level, cfg):\n    values = np.asarray(values, dtype=float)\n    if not len(values) or not np.isfinite(values).all():\n        raise ValueError(\'Некорректные значения для парного bootstrap.\')\n    rng = np.random.default_rng(cfg.bootstrap_seed)\n    means = np.empty(cfg.bootstrap_reps)\n    for start in range(0, cfg.bootstrap_reps, 64):\n        count = min(64, cfg.bootstrap_reps - start)\n        draws = rng.integers(0, len(values), size=(count, len(values)))\n        means[start:start + count] = values[draws].mean(axis=1)\n    alpha = 1 - level\n    low, high = np.quantile(means, [alpha / 2, 1 - alpha / 2])\n    return {\'users\': len(values), \'mean_difference\': float(values.mean()), \'level\': level,\n            \'low\': float(low), \'high\': float(high), \'repetitions\': cfg.bootstrap_reps}\n\ndef make_intervals(data, cfg, root):\n    intervals = []\n    comparisons = [(\'SOFT\', \'ALL\', \'primary\'), (\'SOFT\', \'STEP10\', \'primary\'),\n                   (\'TARGET10\', \'DELETE10\', \'secondary\'), (\'SOFT\', \'DELETE10\', \'secondary\')]\n    metrics_list = [\'ndcg10_high\', \'recall10_high\', \'ndcg10_any\', \'recall10_any\']\n    for split in data:\n        per_user = {}\n        for method in METHODS:\n            frames = [pd.read_csv(Path(root) / \'test_runs\' / run_name(split, method, seed) / \'per_user.csv\').set_index(\'user_id\') for seed in SEEDS]\n            reference = frames[0].index\n            if any(not f.index.equals(reference) for f in frames) or not reference.is_unique:\n                raise ValueError(\'Различается порядок или состав пользователей между seed.\')\n            per_user[method] = sum((f[metrics_list] for f in frames)) / len(SEEDS)\n        for left, right, role in comparisons:\n            if not per_user[left].index.equals(per_user[right].index):\n                raise ValueError(\'Нарушено парное сравнение пользователей.\')\n            for metric in metrics_list:\n                delta = (per_user[left][metric] - per_user[right][metric]).dropna()\n                level = .9875 if role == \'primary\' and metric == \'ndcg10_high\' else .95\n                intervals.append({\'split\': split, \'left\': left, \'right\': right, \'metric\': metric,\n                                  \'role\': \'primary\' if role == \'primary\' and metric == \'ndcg10_high\' else \'descriptive\',\n                                  **bootstrap_interval(delta.to_numpy(), level, cfg)})\n    save_json(Path(root) / \'paired_intervals.json\', intervals)\n    save_csv(Path(root) / \'paired_intervals.csv\', intervals)\n    return intervals\n\ndef plot_results(table, intervals, root):\n    import matplotlib\n    matplotlib.use(\'Agg\')\n    import matplotlib.pyplot as plt\n    from matplotlib.ticker import FuncFormatter, MaxNLocator\n    plt.rcParams.update({\'font.family\': \'DejaVu Sans\', \'font.size\': 9, \'axes.titlesize\': 10,\n                         \'axes.spines.top\': False, \'axes.spines.right\': False, \'svg.fonttype\': \'path\',\n                         \'savefig.facecolor\': \'white\'})\n    def decimal_axis(ax):\n        ax.xaxis.set_major_locator(MaxNLocator(4))\n        ticks = ax.get_xticks()\n        step = abs(float(ticks[1] - ticks[0])) if len(ticks) > 1 else 1.0\n        digits = max(3, int(math.ceil(-math.log10(step) - 1e-9))) if step else 3\n        formatter = (lambda x, _: f\'{x:.{digits}f}\'.replace(\'.\', \',\')) if digits <= 6 else (lambda x, _: f\'{x:.2g}\'.replace(\'.\', \',\'))\n        ax.xaxis.set_major_formatter(FuncFormatter(formatter))\n    colors = [\'#245B8A\', \'#84919C\', \'#B46B30\', \'#7060A2\', \'#267151\']\n    for metric, stem, title in [(\'test_ndcg10_high\', \'quality_overview\', \'NDCG@10: доля просмотра ≥50%\'),\n                                 (\'training_seconds\', \'cost_overview\', \'Обучение, с\')]:\n        fig, axes = plt.subplots(1, 2, figsize=(8.0, 3.5), layout=\'constrained\')\n        for ax, split in zip(axes, [\'main\', \'replication\']):\n            part = table.loc[table.split.eq(split)].set_index(\'method\').loc[METHODS]\n            y = np.arange(len(METHODS))[::-1]\n            for position, method, color in zip(y, METHODS, colors):\n                ax.errorbar(part.loc[method, metric + \'_mean\'], position,\n                            xerr=part.loc[method, metric + \'_std\'], color=color, marker=\'o\', capsize=3, lw=1.1)\n            ax.set(yticks=y, yticklabels=METHODS, ylim=(-.6, len(METHODS)-.4), xlabel=title)\n            ax.set_title(\'Основной срез\' if split == \'main\' else \'Повторение во времени\', loc=\'left\', pad=8)\n            ax.grid(axis=\'x\', color=\'#E1E5E8\', lw=.6)\n            ax.set_axisbelow(True)\n            ax.xaxis.set_major_locator(MaxNLocator(4))\n            if metric.startswith(\'test_\'):\n                decimal_axis(ax)\n        for ext in [\'png\', \'svg\']:\n            fig.savefig(Path(root) / (stem + \'.\' + ext), dpi=320, bbox_inches=\'tight\', pad_inches=.12)\n        plt.close(fig)\n    # A coherent separate figure for the four prespecified primary contrasts.\n    fig, ax = plt.subplots(figsize=(7.2, 3.3), layout=\'constrained\')\n    primary = [r for r in intervals if r[\'role\'] == \'primary\']\n    for i, row in enumerate(primary[::-1]):\n        mean = row[\'mean_difference\']\n        ax.plot([row[\'low\'], row[\'high\']], [i, i], color=\'#267151\', lw=1.2)\n        ax.vlines([row[\'low\'], row[\'high\']], i-.06, i+.06, color=\'#267151\', lw=1.2)\n        ax.plot(mean, i, color=\'#267151\', marker=\'o\')\n    ax.set_yticks(range(len(primary)), [(\'Основной\' if r[\'split\']==\'main\' else \'Повторение\') + \': \' + r[\'left\']+\' − \'+r[\'right\'] for r in primary[::-1]])\n    ax.axvline(0, color=\'#657078\', lw=.8, ls=\'--\')\n    ax.grid(axis=\'x\', color=\'#E1E5E8\', lw=.6)\n    ax.set_xlabel(\'Разность NDCG@10 высокой доли; пользовательский интервал 98,75%\')\n    decimal_axis(ax)\n    for ext in [\'png\', \'svg\']:\n        fig.savefig(Path(root) / (\'primary_differences.\' + ext), dpi=320, bbox_inches=\'tight\', pad_inches=.12)\n    plt.close(fig)\n\ndef report_study(data, cfg, root, frozen, device):\n    check_execution(cfg, frozen, device)\n    root = Path(root)\n    examples, lock = freeze_test(data, cfg, root, frozen)\n    runs = test_models(data, examples, cfg, root, frozen, device, lock)\n    marker = root / \'report_completed.json\'\n    if marker.exists():\n        saved = json.loads(marker.read_text(encoding=\'utf-8\'))\n        if saved[\'plan_hash\'] != frozen[\'plan_hash\']:\n            raise ValueError(\'Итоговый отчёт относится к другой серии.\')\n        for name, digest in saved[\'checksums\'].items():\n            if not (root / name).exists() or file_hash(root / name) != digest:\n                raise ValueError(\'Изменён файл итогового отчёта: \' + name)\n        return pd.read_csv(root / \'results_by_method.csv\')\n    numeric = [c for c in runs.select_dtypes(include=\'number\').columns if c != \'seed\']\n    table = runs.groupby([\'split\', \'method\'])[numeric].agg([\'mean\', \'std\'])\n    table.columns = [\'_\'.join(c) for c in table.columns]\n    table = table.reset_index()\n    table[\'seeds\'] = len(SEEDS)\n    save_csv(root / \'results_by_method.csv\', table)\n    intervals = make_intervals(data, cfg, root)\n    plot_results(table, intervals, root)\n    curves = []\n    for split, method, seed in queue(data):\n        f = pd.read_csv(root / \'runs\' / run_name(split, method, seed) / \'history.csv\')\n        f = f.assign(split=split, method=method, seed=seed)\n        curves.append(f)\n    save_csv(root / \'learning_curves.csv\', pd.concat(curves, ignore_index=True))\n    summary = {\'status\': \'complete_30_model_study\' if not cfg.technical_mode else \'technical_smoke_complete\',\n               \'plan_hash\': frozen[\'plan_hash\'], \'models\': len(runs), \'methods\': METHODS, \'seeds\': SEEDS,\n               \'splits\': list(data), \'config\': asdict(cfg), \'environment\': frozen[\'plan\'][\'environment\'],\n               \'primary_comparisons\': [r for r in intervals if r[\'role\'] == \'primary\'],\n               \'test_manifests\': lock[\'test_manifests\'], \'checkpoint_selection\': \'validation_ndcg10_high\',\n               \'interpretation\': \'conditional_user_intervals; not satisfaction, causal debiasing or online revenue\',\n               \'training_hours_without_validation\': float(runs.training_seconds.sum() / 3600)}\n    save_json(root / \'FINAL_SUMMARY.json\', summary)\n    (root / \'RESULTS_RU.md\').write_text(\'\'\'# Результаты исследования доли просмотра KION\n\nПолная таблица — results_by_method.csv, отдельные модели — results_per_run.csv.\nОсновная метрика — test_ndcg10_high: цели с долей просмотра ≥50%.\nМетрики с суффиксом _any используют все допустимые просмотры; размеры популяций различаются.\nSD отражает три seed. Четыре основных пользовательских интервала имеют уровень 98,75%;\nостальные 95%-интервалы описательные. Интервалы условны на обученных моделях.\nSOFT — заранее заданная эвристика, не доказанная оценка истинного интереса.\nВыбранные по валидации checkpoints всех 30 моделей оценены; тест не выбирает метод.\nСырые ID рекомендаций в per_user.csv внутренние, отображение — data/<split>/selection_catalog.npz.\nВ лёгком архиве нет весов. Сохраните исходные выходы Kaggle с папкой runs.\n\'\'\', encoding=\'utf-8\')\n    required = [\'results_per_run.csv\', \'results_by_method.csv\', \'paired_intervals.json\', \'paired_intervals.csv\',\n                \'learning_curves.csv\', \'FINAL_SUMMARY.json\', \'RESULTS_RU.md\']\n    required += [stem + \'.\' + ext for stem in [\'quality_overview\', \'cost_overview\', \'primary_differences\'] for ext in [\'png\', \'svg\']]\n    save_json(marker, {\'plan_hash\': frozen[\'plan_hash\'], \'checksums\': {name: file_hash(root / name) for name in required}})\n    return table\n\ndef export_results(root):\n    root = Path(root)\n    target = root.parent / (root.name + \'_results.zip\')\n    temporary = target.with_suffix(\'.tmp\')\n    allowed = {\'.json\', \'.csv\', \'.npz\', \'.md\', \'.png\', \'.svg\', \'.py\'}\n    with zipfile.ZipFile(temporary, \'w\', zipfile.ZIP_DEFLATED, compresslevel=6) as z:\n        for path in sorted(root.rglob(\'*\')):\n            if (path.is_file() and path.suffix in allowed and not path.name.startswith(\'candidate_\')\n                    and not any(p.startswith(\'.\') for p in path.relative_to(root).parts)):\n                z.write(path, path.relative_to(root).as_posix())\n    with zipfile.ZipFile(temporary) as z:\n        if z.testzip() is not None:\n            raise ValueError(\'Ошибка проверки экспортированного архива.\')\n    temporary.replace(target)\n    return target\n'
PROGRAM_SHA256 = '5ab07ac369ae3966f3d781ada94a38415eed710ade65541c594c2a4fd3b57509'
if hashlib.sha256(PROGRAM_SOURCE.encode()).hexdigest() != PROGRAM_SHA256:
    raise ValueError("Изменён встроенный код. Для этой серии нужен исходный ноутбук.")
exec(compile(PROGRAM_SOURCE, "<kion-watch-feedback-program>", "exec"))
import matplotlib
try:
    from IPython.display import display, FileLink
except ImportError:
    display = print
    FileLink = lambda path: path
torch.set_num_threads(2)


## 2. Настройки запуска

Научные параметры уже заданы — оставьте CFG прежним. Полная очередь содержит 30 моделей; автоматического расширения поиска нет.

- **RESUME_FROM** обычно пуст. Для новой сессии добавьте сохранённые выходы Kaggle как Input: совместимая папка ищется автоматически. Если нужен конкретный источник, укажите папку с `watch_study_plan.json` и `runs`.
- **DATA_ARCHIVE** обычно пуст: официальный KION скачается автоматически. При желании можно указать уже добавленный архив `data_en.zip`.
- **MAX_RUNS_PER_SESSION = 0** выполняет всю оставшуюся очередь. Положительное число ограничивает только количество новых моделей в текущей сессии; общий план остаётся из 30 моделей.
- Лёгкий ZIP результатов не содержит весов и не заменяет сохранённые выходы для восстановления.

Среда фиксируется при начале новой серии. Если при продолжении изменились версии библиотек или GPU, ноутбук остановится с объяснением. Проверку не отключайте.


In [ ]:
CFG = WatchConfig()
EXPERIMENT_ID = "kion_watch_feedback_50k_S20_v1"
WORK_ROOT = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd() / "watch_feedback_outputs"
RESULTS_DIR = WORK_ROOT / EXPERIMENT_ID
DATA_ARCHIVE = ""  # необязательно: абсолютный путь к официальному data_en.zip
RESUME_FROM = ""   # необязательно: папка с watch_study_plan.json и runs
MAX_RUNS_PER_SESSION = 0
PILOT_STEPS = 200  # короткая проверка скорости; это не научный запуск


## 3. Подготовка фильмов и фиксация протокола

Для каждого среза независимо выбираются 50 000 пользователей с минимум десятью фильмами на обучении. Когорта и каталог общие для всех методов и определяются до фильтрации слабых просмотров. Пропуски процента не считаются нулями.

| Срез | Обучение включительно | Валидация | Тест |
| --- | --- | --- | --- |
| Основной | 01.08.2021 | 02–08.08 | 09–15.08 |
| Повторение во времени | 18.07.2021 | 19–25.07 | 26.07–01.08 |

Цели — известные каталогу фильмы первого наблюдаемого дня будущего периода. День не переносится после исключения неизвестных объектов или пропусков. Из кандидатов исключаются все известные прошлые просмотры, включая удалённые DELETE10. Пользователи с пустой историей DELETE10 остаются в тесте.

Выбор checkpoint — по валидационному NDCG@10 высокой доли, у всех одинаково. Тестовые метрики до завершения всей очереди не рассчитываются. Конфигурация, код, данные и среда фиксируются до обучения.


In [ ]:
CFG.validate()
DEVICE = select_device(CFG)
print("Среда новой серии:", json.dumps(environment(DEVICE), ensure_ascii=False, indent=2))
ARCHIVE = acquire_archive(WORK_ROOT / "kion_data_cache", DATA_ARCHIVE)
FILMS = read_films(ARCHIVE)
DATA = {cut[0]: prepare_split(FILMS, cut, CFG) for cut in CUTS}
FROZEN = make_plan(DATA, CFG, DEVICE, PROGRAM_SHA256)
RESTORED_FROM = restore_folder(RESULTS_DIR, FROZEN, RESUME_FROM)
freeze_plan(RESULTS_DIR, FROZEN, DATA)
source_file = RESULTS_DIR / "implementation.py"
if source_file.exists() and source_file.read_text(encoding="utf-8") != PROGRAM_SOURCE:
    raise ValueError("Сохранённый код серии отличается.")
source_file.write_text(PROGRAM_SOURCE, encoding="utf-8")
print("План:", FROZEN["plan_hash"])
print("Всего моделей:", len(queue(DATA)))
display(pd.DataFrame([{ "срез": k, **v.manifest["stats"] } for k, v in DATA.items()]))
print("Папка результатов и весов:", RESULTS_DIR)


## 4. Короткая техническая проверка и ориентир времени

Выполняются 200 шагов и одна валидация отдельной модели. Её веса и качество не используются в научном сравнении. Печатается грубая оценка времени всех 30 моделей, без подготовки данных, записи checkpoints, тестирования и bootstrap. Это ориентир: нагрузка сервера и способы обработки различаются. При продолжении сохранённая проверка пропускается.


In [ ]:
TIMING = benchmark(DATA, CFG, RESULTS_DIR, FROZEN, DEVICE, PILOT_STEPS)
print("Ориентир обучения и валидации всей серии:", round(TIMING["rough_hours_all_models"], 2), "ч")
print("Подготовка, запись весов, тест и интервалы добавят время.")


## 5. Обучение всей очереди

Готовые совместимые модели пропускаются. Незавершённая продолжается из `last.pt`, который записывается каждые 1000 шагов. При остановке может потребоваться повторить до 1000 незаписанных шагов. Случайные состояния восстанавливаются.

Качество и время сохраняются раздельно. Для TARGET10 нулевой вес не превращает просмотр в отрицательную метку. DELETE10 может сократить число обучающих историй — это записывается, а такие пользователи остаются в общей оценке. Любые изменения правил во время серии запрещены её сохранённым планом.


In [ ]:
TRAINING_COMPLETE = False
try:
    TRAINING_COMPLETE = run_study(DATA, CFG, RESULTS_DIR, FROZEN, DEVICE, MAX_RUNS_PER_SESSION)
except KeyboardInterrupt:
    print("Остановка. Сохраните выходы и повторите этот же ноутбук для продолжения.")
finally:
    PARTIAL_ZIP = export_results(RESULTS_DIR)
    print("Архив текущего состояния:", PARTIAL_ZIP)
print("Все 30 моделей готовы:", TRAINING_COMPLETE)


## 6. Общий тест, интервалы, рисунки и один итоговый ZIP

После завершения всех моделей фиксируются контрольные суммы выбранных checkpoints и общие тестовые цели. Рассчитываются все пять методов на обоих срезах. Тест не используется для выбора удачного метода, seed или бюджета.

Четыре основных сравнения — SOFT − ALL и SOFT − STEP10 по NDCG высокой доли на каждом срезе. Пользовательские интервалы 98,75% учитывают четыре основных сравнения; 5000 повторных выборок выполняются после усреднения по трём seed. SD seed — отдельная величина. Остальные 95%-интервалы описательные. Срезы не объявляются независимыми датасетами.

Результаты любого знака сохраняются. PNG и SVG имеют согласованные панели и не кодируют техническое происхождение точек. Числовые таблицы позволяют затем подготовить окончательные рисунки статьи.

После сообщения о завершении скачайте файл **kion_watch_feedback_50k_S20_v1_results.zip**. Он включает код, планы, таблицы, пользовательские метрики и рекомендации, кривые, интервалы и графики. **Веса best.pt/last.pt остаются в runs внутри исходных выходов Kaggle.** Сохраните их через сохранение версии ноутбука с выходами; с новым Input можно будет продолжить серию. Это особенно важно перед завершением сессии.


In [ ]:
if TRAINING_COMPLETE:
    TABLE = report_study(DATA, CFG, RESULTS_DIR, FROZEN, DEVICE)
    display(TABLE[["split", "method", "test_ndcg10_high_mean", "test_ndcg10_high_std", "test_ndcg10_any_mean", "training_seconds_mean"]])
    print("Исследование завершено: все 30 моделей оценены. Пришлите итоговый ZIP.")
else:
    print("Серия ещё не завершена. Сохраните выходы с весами и повторите ноутбук для продолжения.")
RESULTS_ZIP = export_results(RESULTS_DIR)
print("Архив:", RESULTS_ZIP)
display(FileLink(os.path.relpath(RESULTS_ZIP, Path.cwd())))


## Примечание к научным выводам

Изменяется способ использования зарегистрированной доли просмотра, архитектура фиксирована. SOFT не заявляется оценкой истинного интереса или устранением duration bias. Сравнение TARGET10 и DELETE10 оценивает практические способы обработки, а не чистый причинный эффект: удаление меняет длины и состав окон. Короткие входы DELETE10 обрабатываются явно.

KION уже применялся в другой задаче; эта серия не является независимым подтверждением первой статьи. Ограничения агрегированных взаимодействий, одного датасета, трёх seed и заданного бюджета сохраняются. Даже отсутствие выигрыша перед ALL — результат этой проверки.

Источники: [описание KION](https://arxiv.org/html/2209.00325), [ступенчатые веса в примере RecTools](https://rectools.readthedocs.io/en/stable/examples/tutorials/transformers_advanced_training_guide.html), [D²Co, RecSys 2023](https://arxiv.org/html/2308.08120). Протокол сформулирован до новых тестовых метрик.
